# LeetCode 1239 — Maximum Length of a Concatenated String with Unique Characters bằng Python

Link đề: https://leetcode.com/problems/maximum-length-of-a-concatenated-string-with-unique-characters/ · Độ khó: **Medium** · Chủ đề: Array, String, Backtracking, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def maxLength(self, arr: List[str]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên** — độ dài lớn nhất của chuỗi ghép từ một dãy con của `arr` mà mọi ký tự đều phân biệt.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn một **dãy con** (subsequence) các chuỗi trong `arr`, ghép lại, yêu cầu **không ký tự nào lặp**; tối đa hoá độ dài.
2. Biểu diễn một tập chữ cái thường bằng **bitmask 26 bit** (`1 << (ord(c) - ord('a'))`), kiểm tra giao nhau bằng `&`, hợp bằng `|`, đếm bằng `bit_count`.
3. Viết được vét cạn mọi tập con (`2ⁿ` với `n ≤ 16`), rồi nâng cấp thành **backtracking** có cắt nhánh và **DP trên tập các mask đạt được**.
4. Nhớ bước tiền xử lý hay bị quên: **loại bỏ chuỗi tự nó đã có ký tự lặp** (ví dụ `"aa"`).
5. Biết thêm kỹ thuật **branch and bound** (cắt nhánh theo cận trên) để chạy nhanh hơn trên thực tế.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `arr`. Một chuỗi `s` được tạo bằng cách **ghép một dãy con** của `arr` sao cho `s` có **các ký tự đôi một khác nhau**. Trả về **độ dài lớn nhất có thể** của `s`.

Dãy con là mảng thu được từ `arr` bằng cách xoá một số (hoặc không) phần tử mà giữ nguyên thứ tự. (Với bài này thứ tự không ảnh hưởng độ dài, nên thực chất là chọn một **tập con**.)

**Ví dụ 1:** `arr = ["un","iq","ue"]` → `4` (`"uniq"` hoặc `"ique"`).

**Ví dụ 2:** `arr = ["cha","r","act","ers"]` → `6` (`"chaers"` hoặc `"acters"`).

**Ví dụ 3:** `arr = ["abcdefghijklmnopqrstuvwxyz"]` → `26`.

**Ràng buộc:** `1 <= arr.length <= 16`; `1 <= arr[i].length <= 26`; `arr[i]` chỉ gồm chữ cái thường tiếng Anh.

In [1]:
import itertools
import random
import string
import time
from functools import lru_cache
from typing import List

examples = [
    (["un", "iq", "ue"], 4),
    (["cha", "r", "act", "ers"], 6),
    (["abcdefghijklmnopqrstuvwxyz"], 26),
]
for arr, want in examples:
    print(arr, "->", want)

['un', 'iq', 'ue'] -> 4
['cha', 'r', 'act', 'ers'] -> 6
['abcdefghijklmnopqrstuvwxyz'] -> 26


## 3. Kiến thức nền

### 3.1 Kiểm tra chuỗi có ký tự phân biệt bằng `set`

Chuỗi `s` có mọi ký tự khác nhau ⇔ `len(set(s)) == len(s)`.

In [2]:
for s in ["uniq", "unue", "aa", "abc", ""]:
    print(repr(s), len(set(s)) == len(s))

'uniq' True
'unue' False
'aa' False
'abc' True
'' True


### 3.2 Bitmask cho tập chữ cái

Có đúng 26 chữ cái thường → mỗi tập chữ cái là một số nguyên 26 bit: bit `k` bật ⇔ chữ cái thứ `k` (`'a'` là 0) có mặt.

- Tạo mask: `m |= 1 << (ord(c) - 97)`.
- Hai tập **giao nhau** ⇔ `m1 & m2 != 0`.
- Hợp hai tập: `m1 | m2`.
- Số phần tử: `m.bit_count()` (Python ≥ 3.10) hoặc `bin(m).count('1')`.

In [3]:
def to_mask(s):
    """Trả về mask của s, hoặc -1 nếu s có ký tự lặp."""
    m = 0
    for c in s:
        b = 1 << (ord(c) - 97)
        if m & b:
            return -1
        m |= b
    return m


for s in ["un", "iq", "ue", "aa"]:
    m = to_mask(s)
    print(f"{s!r:6} mask={m:>8} bin={m:026b}" if m >= 0 else f"{s!r:6} có ký tự lặp -> -1")
mu, mi, me = to_mask("un"), to_mask("iq"), to_mask("ue")
print("un & iq =", mu & mi, "| un & ue =", mu & me, "| |un ∪ iq| =", (mu | mi).bit_count())

'un'   mask= 1056768 bin=00000100000010000000000000
'iq'   mask=   65792 bin=00000000010000000100000000
'ue'   mask= 1048592 bin=00000100000000000000010000
'aa'   có ký tự lặp -> -1
un & iq = 0 | un & ue = 1048576 | |un ∪ iq| = 4


### 3.3 Duyệt mọi tập con bằng số nhị phân

Với `n` phần tử, mỗi số `S` từ `0` đến `2ⁿ − 1` là một tập con: phần tử `i` được chọn ⇔ bit `i` của `S` bật. `itertools.combinations` là cách khác để liệt kê tập con theo kích thước.

In [4]:
arr = ["un", "iq", "ue"]
for S in range(1 << len(arr)):
    chosen = [arr[i] for i in range(len(arr)) if S >> i & 1]
    print(f"S={S:03b} -> {chosen}")

S=000 -> []
S=001 -> ['un']
S=010 -> ['iq']
S=011 -> ['un', 'iq']
S=100 -> ['ue']
S=101 -> ['un', 'ue']
S=110 -> ['iq', 'ue']
S=111 -> ['un', 'iq', 'ue']


### 3.4 Backtracking (quay lui)

Backtracking = thử từng lựa chọn, đi sâu, rồi **hoàn tác** để thử lựa chọn khác. Mẫu chung "chọn hoặc không chọn phần tử `i`":

```python
def dfs(i, state):
    if i == n: cập nhật đáp án; return
    dfs(i + 1, state)                    # không chọn
    if có thể chọn: dfs(i + 1, state + arr[i])   # chọn
```

Khi `state` là số nguyên (bitmask), ta truyền giá trị mới vào lời gọi nên **không cần hoàn tác** thủ công.

In [5]:
def all_choices(items):
    out = []

    def dfs(i, cur):
        if i == len(items):
            out.append(cur)
            return
        dfs(i + 1, cur)
        dfs(i + 1, cur + items[i])

    dfs(0, "")
    return out


print(all_choices(["a", "b", "c"]))

['', 'c', 'b', 'bc', 'a', 'ac', 'ab', 'abc']


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `arr = ["cha","r","act","ers"]`.

- `"cha"` và `"act"` chung `a`, `c` → không đi cùng nhau.
- `"r"` và `"ers"` chung `r`.
- Các tập hợp lệ lớn: `{"cha","ers"}` → `chaers` (6); `{"act","ers"}` → `acters` (6); `{"cha","r"}` → 4; `{"act","r"}` → 4.
- Thử thêm `"r"` vào `{"cha","ers"}` → trùng `r` với `"ers"` → không được.

Vậy đáp án `6`. Quan sát: đây là bài **chọn tập con** với ràng buộc "đôi một rời nhau" (pairwise disjoint), mục tiêu tối đa tổng độ dài — họ hàng của bài toán xếp ba lô / tập độc lập, nhưng `n ≤ 16` nên `2¹⁶ = 65536` tập con là duyệt được.

In [6]:
arr = ["cha", "r", "act", "ers"]
best = []
for S in range(1 << len(arr)):
    s = "".join(arr[i] for i in range(len(arr)) if S >> i & 1)
    if len(set(s)) == len(s):
        best.append((len(s), s))
best.sort(reverse=True)
print(best[:5])

[(6, 'chaers'), (6, 'acters'), (4, 'ract'), (4, 'char'), (3, 'ers')]


## 5. Cách 1 — Brute Force / baseline

Duyệt mọi tập con `S` (`2ⁿ` cái), ghép chuỗi, kiểm tra bằng `set`.

In [7]:
class SolutionBrute:
    def maxLength(self, arr):
        n = len(arr)
        ans = 0
        for S in range(1 << n):
            s = "".join(arr[i] for i in range(n) if S >> i & 1)
            if len(set(s)) == len(s):
                ans = max(ans, len(s))
        return ans


for arr, want in examples:
    print(SolutionBrute().maxLength(arr), want)

4 4
6 6
26 26


**Độ phức tạp:** `O(2ⁿ · L)` với `L` = tổng độ dài các chuỗi (≤ `16 · 26 = 416`) → khoảng `65536 · 416 ≈ 2.7 · 10⁷` thao tác ký tự trong trường hợp xấu. Chấp nhận được với `n ≤ 16` nhưng chậm trong Python.

**Nút thắt:** (1) ghép và kiểm tra lại chuỗi từ đầu cho **mỗi** tập con; (2) không cắt nhánh: một khi `{"cha","act"}` đã xung đột thì mọi tập chứa cả hai đều vô ích, nhưng ta vẫn duyệt.

## 6. Bước đột phá

**Insight 1 — Bitmask:** thay mỗi chuỗi bằng mask 26 bit. Kiểm tra "ghép thêm `arr[i]` có còn hợp lệ" chỉ là `cur & m[i] == 0` — **O(1)**. Độ dài = `bit_count()` của mask (vì ký tự phân biệt).

**Insight 2 — Tiền xử lý:** chuỗi nào **tự nó** đã có ký tự lặp (như `"aa"`, `"abca"`) thì không bao giờ dùng được → loại bỏ ngay.

**Insight 3 — Cắt nhánh:** xây tập con dần dần; nếu thêm `arr[i]` gây xung đột thì **không** đi tiếp nhánh đó. Mọi tập con mở rộng từ một trạng thái xung đột đều bị cắt.

**Invariant:** mọi trạng thái `cur` trong quá trình tìm kiếm đều là mask của một chuỗi ghép **hợp lệ**.

**Quy tắc:** nếu `cur & m == 0` thì có thể chọn, trạng thái mới `cur | m`; ngược lại bỏ qua.

In [8]:
masks = [to_mask(s) for s in ["cha", "r", "act", "ers", "aa"]]
print(masks)
masks = [m for m in masks if m >= 0]
print("sau khi loại chuỗi tự lặp:", masks)
print("cha & act =", masks[0] & masks[2], "-> xung đột;", "cha & ers =", masks[0] & masks[3], "-> ghép được")

[133, 131072, 524293, 393232, -1]
sau khi loại chuỗi tự lặp: [133, 131072, 524293, 393232]
cha & act = 5 -> xung đột; cha & ers = 0 -> ghép được


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Backtracking với bitmask

`dfs(i, cur)`: xét chuỗi thứ `i`, đang có mask `cur`. Luôn có lựa chọn "bỏ qua"; nếu không xung đột thì có thêm lựa chọn "chọn".

In [9]:
class SolutionBacktrack:
    def maxLength(self, arr):
        masks = [m for m in map(to_mask, arr) if m >= 0]
        n = len(masks)
        best = 0

        def dfs(i, cur):
            nonlocal best
            if i == n:
                best = max(best, cur.bit_count())
                return
            dfs(i + 1, cur)                       # bỏ qua masks[i]
            if cur & masks[i] == 0:               # chọn được
                dfs(i + 1, cur | masks[i])

        dfs(0, 0)
        return best


for arr, want in examples:
    print(SolutionBacktrack().maxLength(arr), want)

4 4
6 6
26 26


**Độ phức tạp:** worst case vẫn `O(2ⁿ)` (khi mọi chuỗi rời nhau — nhưng khi đó có tối đa 26 chữ cái chia cho các chuỗi, nên số chuỗi rời nhau thực tế bị chặn). Mỗi nút `O(1)`. Bộ nhớ: độ sâu đệ quy `n ≤ 16`.

### 7.2 Cách 3 — DP trên tập các mask đạt được (lặp)

Giữ danh sách `dp` gồm mọi mask **hợp lệ** đạt được từ các chuỗi đã xét (ban đầu `[0]`). Với mỗi chuỗi mới `m`, với mọi `x` trong `dp` mà `x & m == 0`, thêm `x | m`. Đáp án là `max(bit_count)`.

In [10]:
class SolutionDP:
    def maxLength(self, arr):
        dp = [0]
        for s in arr:
            m = to_mask(s)
            if m < 0:
                continue
            dp += [x | m for x in dp if x & m == 0]
        return max(x.bit_count() for x in dp)


for arr, want in examples:
    print(SolutionDP().maxLength(arr), want)

4 4
6 6
26 26


Chú ý `dp += [... for x in dp ...]`: list comprehension được tính **xong** trên `dp` cũ rồi mới nối, nên mỗi chuỗi chỉ được dùng một lần (giống duyệt ngược trong ba lô 0/1).

**Độ phức tạp:** `|dp|` ≤ số tập con hợp lệ ≤ `2ⁿ`; tổng `O(n · |dp|)`. Có thể dùng `set` để khử mask trùng (hai tập con khác nhau cho cùng mask).

### 7.3 Cách 4 — Duyệt tập con bằng bitmask "từ nhỏ đến lớn" (DP theo tập con)

`union[S]` = mask hợp của các chuỗi trong `S`, hoặc `-1` nếu xung đột. Tính từ `S` bỏ bit thấp nhất: `union[S] = union[S ^ low] | m[low]` nếu hai bên rời nhau. Mỗi tập con tốn `O(1)` thay vì `O(L)` như Cách 1.

In [11]:
class SolutionSubsetDP:
    def maxLength(self, arr):
        masks = [m for m in map(to_mask, arr) if m >= 0]
        n = len(masks)
        union = [0] * (1 << n)
        best = 0
        for S in range(1, 1 << n):
            low = S & -S                       # bit thấp nhất
            i = low.bit_length() - 1
            prev = union[S ^ low]
            if prev < 0 or prev & masks[i]:
                union[S] = -1
            else:
                union[S] = prev | masks[i]
                best = max(best, union[S].bit_count())
        return best


for arr, want in examples:
    print(SolutionSubsetDP().maxLength(arr), want)

4 4
6 6
26 26


**Độ phức tạp:** `O(2ⁿ)` thời gian và bộ nhớ (mảng `union` dài 65536). Đơn giản, không đệ quy, nhưng không cắt nhánh.

### 7.4 Cách 5 — Backtracking + branch and bound

Thêm **cận trên**: nếu `bit_count(cur) + (tổng độ dài các chuỗi còn lại)` ≤ `best` hoặc `bit_count(cur | tất cả mask còn lại)` ≤ `best` thì nhánh này không thể tốt hơn → cắt. Sắp xếp chuỗi dài trước để sớm có `best` lớn. Còn có thể dừng ngay khi `best == 26`.

In [12]:
class SolutionBnB:
    def maxLength(self, arr):
        masks = sorted({m for m in map(to_mask, arr) if m >= 0}, key=lambda m: -m.bit_count())
        n = len(masks)
        suffix_or = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            suffix_or[i] = suffix_or[i + 1] | masks[i]
        best = 0

        def dfs(i, cur):
            nonlocal best
            c = cur.bit_count()
            if c > best:
                best = c
            if i == n or best == 26:
                return
            if (cur | suffix_or[i]).bit_count() <= best:   # cận trên
                return
            if cur & masks[i] == 0:
                dfs(i + 1, cur | masks[i])
            dfs(i + 1, cur)

        dfs(0, 0)
        return best


for arr, want in examples:
    print(SolutionBnB().maxLength(arr), want)

4 4
6 6
26 26


Ghi chú: ta dùng `set` để khử chuỗi trùng mask (ví dụ `"ab"` và `"ba"` — chọn được nhiều nhất một cái nên giữ một là đủ).

**Độ phức tạp:** worst case vẫn hàm mũ nhưng thực tế nhanh hơn nhiều nhờ cắt theo cận.

## 8. TRACE từng bước

Trace Cách 3 (DP danh sách mask) trên `arr = ["un","iq","ue"]`, hiển thị mask dưới dạng tập chữ cái:

| Bước | Chuỗi | Mask mới thêm vào `dp` | `dp` (dạng chữ) |
|---|---|---|---|
| 0 | — | — | {∅} |
| 1 | `un` | `un` | {∅, nu} |
| 2 | `iq` | `iq`, `iqnu` | {∅, nu, iq, inqu} |
| 3 | `ue` | `ue`, `eiqu` (từ `iq`); `un` xung đột | {∅, nu, iq, inqu, eu, eiqu} |

Đáp án `max |mask| = 4`.

In [13]:
def letters(m):
    return "".join(chr(97 + k) for k in range(26) if m >> k & 1) or "∅"


def trace_dp(arr):
    dp = [0]
    print("bước 0: dp =", [letters(x) for x in dp])
    for step, s in enumerate(arr, 1):
        m = to_mask(s)
        if m < 0:
            print(f"bước {step}: {s!r} có ký tự lặp -> bỏ")
            continue
        new = [x | m for x in dp if x & m == 0]
        clash = [letters(x) for x in dp if x & m]
        dp += new
        print(f"bước {step}: {s!r} thêm {[letters(x) for x in new]} | xung đột với {clash}")
        print(f"         dp = {[letters(x) for x in dp]}")
    ans = max(x.bit_count() for x in dp)
    print("đáp án:", ans)
    return ans


trace_dp(["un", "iq", "ue"])

bước 0: dp = ['∅']
bước 1: 'un' thêm ['nu'] | xung đột với []
         dp = ['∅', 'nu']
bước 2: 'iq' thêm ['iq', 'inqu'] | xung đột với []
         dp = ['∅', 'nu', 'iq', 'inqu']
bước 3: 'ue' thêm ['eu', 'eiqu'] | xung đột với ['nu', 'inqu']
         dp = ['∅', 'nu', 'iq', 'inqu', 'eu', 'eiqu']
đáp án: 4


4

Trace thêm một input có chuỗi tự lặp để thấy bước tiền xử lý:

In [14]:
trace_dp(["aa", "bb", "ab", "cd"])

bước 0: dp = ['∅']
bước 1: 'aa' có ký tự lặp -> bỏ
bước 2: 'bb' có ký tự lặp -> bỏ
bước 3: 'ab' thêm ['ab'] | xung đột với []
         dp = ['∅', 'ab']
bước 4: 'cd' thêm ['cd', 'abcd'] | xung đột với []
         dp = ['∅', 'ab', 'cd', 'abcd']
đáp án: 4


4

## 9. Code nộp LeetCode

Chọn Cách 3 (DP danh sách mask): ngắn, lặp, dễ đúng.

In [15]:
from typing import List


class Solution:
    def maxLength(self, arr: List[str]) -> int:
        dp = [0]
        for s in arr:
            m = 0
            for c in s:
                b = 1 << (ord(c) - 97)
                if m & b:          # chuỗi tự có ký tự lặp -> bỏ
                    m = -1
                    break
                m |= b
            if m < 0:
                continue
            dp += [x | m for x in dp if x & m == 0]
        return max(bin(x).count("1") for x in dp)


for arr, want in examples:
    assert Solution().maxLength(arr) == want
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


Dùng `bin(x).count("1")` thay `bit_count()` để chạy được cả trên Python cũ hơn 3.10.

## 10. Vì sao đúng

**Bổ đề:** sau khi xử lý `k` chuỗi đầu, `dp` chứa đúng mask của **mọi** tập con hợp lệ (đôi một rời, từng chuỗi không tự lặp) của `k` chuỗi đó (có thể lặp mask).

- `k = 0`: chỉ có tập rỗng, mask `0` ✓.
- Bước `k → k + 1` với chuỗi `m`: mỗi tập con hợp lệ của `k + 1` chuỗi hoặc **không chứa** chuỗi mới (đã có trong `dp` cũ), hoặc **chứa** nó — khi đó phần còn lại là tập hợp lệ `T` của `k` chuỗi đầu với `mask(T) & m == 0`, và ta thêm đúng `mask(T) | m`. Ngược lại, mọi mask được thêm đều từ một tập hợp lệ cộng chuỗi rời với nó → hợp lệ. Chuỗi tự lặp không thuộc tập hợp lệ nào nên bỏ qua là đúng.

**Độ dài:** chuỗi ghép hợp lệ có các ký tự phân biệt, nên độ dài = số bit 1 trong mask. Thứ tự ghép (dãy con giữ thứ tự) không ảnh hưởng độ dài, nên chỉ cần xét tập con.

Do đó `max(bit_count)` trên `dp` cuối là đáp án. Backtracking (Cách 2) duyệt cùng cây lựa chọn nhưng theo chiều sâu; branch and bound chỉ bỏ những nhánh **chứng minh được** không vượt `best`.

## 11. Time / Space Complexity

Đặt `n = len(arr) ≤ 16`, `L = tổng độ dài ≤ 416`, `V` = số tập con hợp lệ (`V ≤ 2ⁿ`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(2ⁿ · L)` | `O(L)` |
| 2. Backtracking | `O(L + số nút) ≤ O(L + 2ⁿ)` | `O(n)` stack |
| 3. DP danh sách mask | `O(L + n · V)` | `O(V)` |
| 4. DP theo tập con | `O(L + 2ⁿ)` | `O(2ⁿ)` |
| 5. Branch and bound | worst `O(L + 2ⁿ)`, thực tế nhỏ hơn nhiều | `O(n)` |

Nhận xét thực tế: vì chỉ có 26 chữ cái, các tập con hợp lệ có tổng độ dài ≤ 26, nên `V` thường nhỏ hơn `2ⁿ` rất nhiều.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [16]:
ALL = [SolutionBrute, SolutionBacktrack, SolutionDP, SolutionSubsetDP, SolutionBnB, Solution]


def check(arr, want=None):
    outs = [cls().maxLength(arr) for cls in ALL]
    assert len(set(outs)) == 1, (arr, outs)
    if want is not None:
        assert outs[0] == want, (arr, outs[0], want)
    return outs[0]


for arr, want in examples:
    check(arr, want)

check(["aa", "bb"], 0)                 # mọi chuỗi tự lặp -> chuỗi rỗng
check(["a"], 1)
check(["ab", "ba", "cd"], 4)           # "ab" và "ba" trùng mask
check(["abc", "def", "abcdef"], 6)
check(["aa", "abc"], 3)
check(["a", "abc", "d", "de", "def"], 6)
check([c for c in string.ascii_lowercase][:16], 16)
print("edge cases OK")

edge cases OK


### 12.2 Phản ví dụ cho chiến lược tham lam "chọn chuỗi dài nhất trước"

Phản ví dụ: `["abcd", "aefg", "bhij"]` — cả ba dài bằng nhau; tham lam lấy `"abcd"` (4) trước, sau đó `"aefg"` trùng `a`, `"bhij"` trùng `b` nên không thêm được gì → 4. Tối ưu là `"aefg" + "bhij"` (8): chuỗi "đẹp" nhất lại chặn hai chuỗi khác.

In [17]:
def greedy_longest_first(arr):
    cur = 0
    for s in sorted(arr, key=len, reverse=True):
        m = to_mask(s)
        if m >= 0 and cur & m == 0:
            cur |= m
    return cur.bit_count()


arr = ["abcd", "aefg", "bhij"]
print("tham lam:", greedy_longest_first(arr), "| đúng:", Solution().maxLength(arr))
assert greedy_longest_first(arr) < Solution().maxLength(arr) == 8

tham lam: 4 | đúng: 8


### 12.3 Ngẫu nhiên có seed: đối chiếu mọi cách với brute force

In [18]:
def random_arr(rng, n, alpha, maxlen):
    letters_ = string.ascii_lowercase[:alpha]
    return ["".join(rng.choice(letters_) for _ in range(rng.randint(1, maxlen))) for _ in range(n)]


rng = random.Random(1239)
for _ in range(400):
    n = rng.randint(1, 10)
    arr = random_arr(rng, n, rng.choice([4, 8, 15, 26]), rng.randint(1, 6))
    check(arr)
print("400 test ngẫu nhiên OK")

400 test ngẫu nhiên OK


### 12.4 Test giới hạn: `n = 16`

Trường hợp xấu cho brute force: 16 chuỗi, mỗi chuỗi 1 chữ cái khác nhau (mọi tập con đều hợp lệ) và 16 chuỗi dài 26.

In [19]:
cases = [
    list(string.ascii_lowercase[:16]),
    ["".join(random.Random(i).sample(string.ascii_lowercase, 26)) for i in range(16)],
    ["".join(random.Random(i).sample(string.ascii_lowercase, 3)) for i in range(16)],
]
for arr in cases:
    row = []
    for cls in ALL:
        t = time.perf_counter()
        r = cls().maxLength(arr)
        row.append(f"{cls.__name__}={r} ({(time.perf_counter() - t) * 1000:.0f}ms)")
    print(" | ".join(row))
    check(arr)

SolutionBrute=16 (87ms) | SolutionBacktrack=16 (12ms) | SolutionDP=16 (6ms) | SolutionSubsetDP=16 (14ms) | SolutionBnB=16 (0ms) | Solution=16 (15ms)


SolutionBrute=26 (178ms) | SolutionBacktrack=26 (0ms) | SolutionDP=26 (0ms) | SolutionSubsetDP=26 (5ms) | SolutionBnB=26 (0ms) | Solution=26 (0ms)


SolutionBrute=18 (110ms) | SolutionBacktrack=18 (0ms) | SolutionDP=18 (0ms) | SolutionSubsetDP=18 (5ms) | SolutionBnB=18 (0ms) | Solution=18 (0ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên loại chuỗi tự có ký tự lặp

In [20]:
def mask_no_check(s):
    m = 0
    for c in s:
        m |= 1 << (ord(c) - 97)       # "aa" -> mask của "a", mất dấu lặp
    return m


def wrong_no_self_check(arr):
    dp = [0]
    for s in arr:
        m = mask_no_check(s)
        dp += [x | m for x in dp if x & m == 0]
    return max(x.bit_count() for x in dp)


arr = ["aa", "b"]
print("sai :", wrong_no_self_check(arr), "| đúng:", Solution().maxLength(arr))

sai : 2 | đúng: 1


**Vì sao sai:** mask của `"aa"` giống `"a"` nên code tưởng `"aa" + "b"` hợp lệ (độ dài theo bit là 2), trong khi `"aab"` có `a` lặp; đáp án đúng là `1` (`"b"`). **Sửa:** khi dựng mask, nếu bit đã bật thì đánh dấu chuỗi không dùng được.

### Lỗi 2 — Cộng độ dài chuỗi nhưng kiểm tra trùng sai (dùng `|` mà không kiểm tra `&`)

In [21]:
def wrong_or_only(arr):
    best = 0
    dp = [(0, 0)]                      # (mask, length)
    for s in arr:
        m = to_mask(s)
        if m < 0:
            continue
        dp += [(x | m, l + len(s)) for x, l in dp]   # SAI: không kiểm tra x & m
    return max(l for _, l in dp)


arr = ["un", "iq", "ue"]
print("sai :", wrong_or_only(arr), "| đúng:", Solution().maxLength(arr))

sai : 6 | đúng: 4


**Vì sao sai:** ghép `"un" + "ue"` có `u` lặp nhưng vẫn được tính độ dài 6. **Sửa:** chỉ ghép khi `x & m == 0`.

### Lỗi 3 — Duyệt `dp` trong lúc đang thêm vào nó (dùng một chuỗi nhiều lần)

In [22]:
def wrong_inplace(arr):
    dp = [0]
    for s in arr:
        m = to_mask(s)
        if m < 0:
            continue
        for x in dp:                   # SAI: dp đang dài ra trong vòng lặp
            if x & m == 0:
                dp.append(x | m)
    return max(x.bit_count() for x in dp), len(dp)


print("sai : (đáp án, |dp|) =", wrong_inplace(["ab", "cd"]))
print("đúng: đáp án =", Solution().maxLength(["ab", "cd"]))

sai : (đáp án, |dp|) = (4, 4)
đúng: đáp án = 4


**Vì sao sai (ở đây là lãng phí):** khi vòng lặp gặp phần tử vừa thêm `x | m`, nó thử ghép `m` lần nữa — may là luôn xung đột nên không ra kết quả sai, nhưng nếu `m = 0` (chuỗi rỗng, không có trong đề) sẽ **lặp vô hạn**; và tư duy "sửa list khi đang duyệt" là nguồn lỗi kinh điển. **Sửa:** tính danh sách mới trước rồi mới nối: `dp += [x | m for x in dp if x & m == 0]`.

### Lỗi 4 — Tham lam theo độ dài

Đã có phản ví dụ ở 12.2: `["abcd","aefg","bhij"]` → tham lam ra 4, đáp án 8. **Sửa:** phải xét tổ hợp (backtracking/DP).

### Lỗi 5 — Dùng `set` dùng chung giữa các nhánh backtracking mà không hoàn tác

In [23]:
def wrong_shared_set(arr):
    used = set()
    best = 0

    def dfs(i, length):
        nonlocal best
        best = max(best, length)
        for j in range(i, len(arr)):
            s = arr[j]
            if len(set(s)) == len(s) and not (used & set(s)):
                used.update(s)
                dfs(j + 1, length + len(s))
                # SAI: quên used.difference_update(s)

    dfs(0, 0)
    return best


arr = ["abc", "abd", "e"]
print("sai :", wrong_shared_set(arr), "| đúng:", Solution().maxLength(arr))

sai : 4 | đúng: 4


**Vì sao sai:** sau nhánh chọn `"abc"`, các chữ `a, b, c` vẫn nằm trong `used`, nên các nhánh khác dùng tới `a`, `b` hoặc `c` bị chặn oan. Với `["abc", "abd", "e"]` kết quả tình cờ vẫn đúng (4), vì vậy ta cho máy tìm một phản ví dụ ngẫu nhiên (seed cố định):

In [24]:
rng = random.Random(3)
found = None
for _ in range(2000):
    arr = random_arr(rng, rng.randint(2, 5), 5, 3)
    if wrong_shared_set(arr) != Solution().maxLength(arr):
        found = arr
        break
print("phản ví dụ:", found, "| sai:", wrong_shared_set(found), "| đúng:", Solution().maxLength(found))

phản ví dụ: ['aab', 'aca', 'de', 'ddd', 'edb'] | sai: 2 | đúng: 3


**Sửa:** truyền trạng thái bằng **giá trị** (bitmask số nguyên, như Cách 2) hoặc hoàn tác `used.difference_update(s)` sau khi gọi đệ quy.

## 14. Mẹo & nhận dạng pattern

- `n ≤ 16` (hoặc ≤ 20) → gợi ý mạnh rằng **duyệt `2ⁿ` tập con** là chấp nhận được.
- "Ký tự phân biệt", "chữ cái thường" → **bitmask 26 bit**; giao nhau `&`, hợp `|`, đếm `bit_count`.
- "Chọn tập con đôi một rời nhau, tối đa tổng" → backtracking chọn/không chọn, hoặc DP trên danh sách trạng thái đạt được.
- Luôn **tiền xử lý** loại phần tử tự vi phạm điều kiện.
- Muốn nhanh hơn trên thực tế: sắp xếp phần tử "giá trị lớn" trước + **cận trên** để cắt nhánh (branch and bound).
- Bài cùng họ: 78 Subsets, 1255 Maximum Score Words Formed by Letters, 1879 Minimum XOR Sum of Two Arrays (DP bitmask), 318 Maximum Product of Word Lengths (bitmask chữ cái).

## 15. Biến thể

1. **Trả về chính chuỗi dài nhất** (không chỉ độ dài) → lưu thêm danh sách chỉ số trong `dp`.
2. **Đếm số cách** chọn tập con cho chuỗi dài nhất / cho mọi chuỗi hợp lệ.
3. **Mỗi ký tự được dùng tối đa `k` lần** → thay bitmask bằng vector đếm 26 chiều.
4. **Có trọng số** (mỗi chuỗi có điểm riêng) → DP tương tự, tối đa tổng điểm thay vì số bit (giống 1255).
5. **LeetCode 318 — Maximum Product of Word Lengths:** chọn **hai** từ rời nhau tối đa tích độ dài → bitmask + duyệt cặp.

Demo biến thể 1 (trả về chuỗi) và 5 (LeetCode 318):

In [25]:
def longest_concat(arr):
    dp = [(0, "")]
    for s in arr:
        m = to_mask(s)
        if m < 0:
            continue
        dp += [(x | m, t + s) for x, t in dp if x & m == 0]
    return max(dp, key=lambda p: len(p[1]))[1]


def max_product(words):
    ms = [(to_mask_any(w), len(w)) for w in words]
    return max([a[1] * b[1] for i, a in enumerate(ms) for b in ms[i + 1:] if a[0] & b[0] == 0], default=0)


def to_mask_any(s):
    m = 0
    for c in s:
        m |= 1 << (ord(c) - 97)
    return m


print(longest_concat(["cha", "r", "act", "ers"]))
print(max_product(["abcw", "baz", "foo", "bar", "xtfn", "abcdef"]))   # 16 (abcw * xtfn)

chaers
16


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Mọi tập con, ghép + `set` | `O(2ⁿ · L)` | `O(L)` | Tham chiếu kiểm thử |
| 2. Backtracking bitmask | Chọn/không chọn, cắt khi xung đột | `O(2ⁿ)` worst | `O(n)` | Phổ biến, dễ mở rộng |
| 3. DP danh sách mask | Mở rộng mọi mask hợp lệ | `O(n · V)` | `O(V)` | Code nộp ngắn gọn |
| 4. DP theo tập con | `union[S]` từ `S` bỏ bit thấp | `O(2ⁿ)` | `O(2ⁿ)` | Khi thích lặp thuần, không đệ quy |
| 5. Branch and bound | Backtracking + cận trên + sort | worst `O(2ⁿ)`, thực tế nhanh | `O(n)` | Khi `n` lớn hơn (ví dụ 25–30) |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `Solution().maxLength(["a", "b", "ab", "abc"])`?

**Bài 2 (tính tay):** với `arr = ["ab", "cd", "ac", "bd"]`, liệt kê các tập con hợp lệ có độ dài 4.

**Bài 3 (điền TODO):** viết `count_max_sets(arr)` trả về **số tập con** đạt độ dài lớn nhất.

**Bài 4 (tự viết lại):** viết `max_length_backtrack(arr)` bằng backtracking dạng "vòng for từ `i`" (mỗi lời gọi thử thêm một chuỗi `j ≥ i`).

**Bài 5 (mini challenge):** viết `max_length_k(arr, k)`: mỗi ký tự được xuất hiện **tối đa `k` lần** trong chuỗi ghép.

In [26]:
def count_max_sets(arr):
    # TODO: DP lưu (mask, số tập con) hoặc duyệt tập con
    raise NotImplementedError


def max_length_backtrack(arr):
    # TODO: dfs(i, cur): thử mọi j >= i
    raise NotImplementedError


def max_length_k(arr, k):
    # TODO: trạng thái là tuple đếm 26 chữ cái
    raise NotImplementedError


# Test gợi ý:
# assert count_max_sets(["un", "iq", "ue"]) == 2
# assert max_length_backtrack(["cha", "r", "act", "ers"]) == 6
# assert max_length_k(["aa", "ab"], 2) == 2 and max_length_k(["aa", "ab"], 3) == 4

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: liệt kê mọi tập con không xung đột.
- Mức 2: `"abc"` chiếm cả `a, b, c`; không ghép thêm được gì.
- Mức 3: Bài 2: các cặp rời nhau là `{ab, cd}` và `{ac, bd}`.

**Bài 3–4:**
- Mức 1: Bài 3: duyệt mọi tập con (brute) là đủ với `n ≤ 16`.
- Mức 2: tính độ dài lớn nhất trước, rồi đếm tập con đạt độ dài đó.
- Mức 3: Bài 4: `dfs(i, cur)`: cập nhật `best`, rồi `for j in range(i, n): if cur & m[j] == 0: dfs(j + 1, cur | m[j])`.

**Bài 5:**
- Mức 1: bitmask không còn đủ; dùng mảng đếm 26 phần tử.
- Mức 2: chọn chuỗi `s` được nếu `cnt[c] + s.count(c) ≤ k` với mọi `c`.
- Mức 3: backtracking chọn/không chọn với `cnt` dạng tuple (bất biến) để khỏi hoàn tác.

## 19. Đáp án tham khảo

**Bài 1:** `3` — chọn `"abc"`; các tổ hợp khác như `"a" + "b"` hay `"ab"` chỉ dài 2. **Bài 2:** `{ab, cd}` và `{ac, bd}`.

In [27]:
assert Solution().maxLength(["a", "b", "ab", "abc"]) == 3
arr = ["ab", "cd", "ac", "bd"]
sets4 = [S for S in range(16) if len(s := "".join(arr[i] for i in range(4) if S >> i & 1)) == 4 and len(set(s)) == 4]
assert sorted([[arr[i] for i in range(4) if S >> i & 1] for S in sets4]) == [["ab", "cd"], ["ac", "bd"]]
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [28]:
def count_max_sets(arr):
    n = len(arr)
    best, cnt = -1, 0
    for S in range(1 << n):
        s = "".join(arr[i] for i in range(n) if S >> i & 1)
        if len(set(s)) == len(s):
            if len(s) > best:
                best, cnt = len(s), 1
            elif len(s) == best:
                cnt += 1
    return cnt


def max_length_backtrack(arr):
    masks = [m for m in map(to_mask, arr) if m >= 0]
    best = 0

    def dfs(i, cur):
        nonlocal best
        best = max(best, cur.bit_count())
        for j in range(i, len(masks)):
            if cur & masks[j] == 0:
                dfs(j + 1, cur | masks[j])

    dfs(0, 0)
    return best


def max_length_k(arr, k):
    best = 0
    vecs = []
    for s in arr:
        v = [0] * 26
        for c in s:
            v[ord(c) - 97] += 1
        vecs.append(tuple(v))

    def dfs(i, cnt, length):
        nonlocal best
        best = max(best, length)
        if i == len(vecs):
            return
        dfs(i + 1, cnt, length)
        new = tuple(a + b for a, b in zip(cnt, vecs[i]))
        if max(new) <= k:
            dfs(i + 1, new, length + len(arr[i]))

    dfs(0, (0,) * 26, 0)
    return best


assert count_max_sets(["un", "iq", "ue"]) == 2
assert count_max_sets(["cha", "r", "act", "ers"]) == 2
assert max_length_backtrack(["cha", "r", "act", "ers"]) == 6
assert max_length_k(["aa", "ab"], 2) == 2 and max_length_k(["aa", "ab"], 3) == 4
rng = random.Random(11)
for _ in range(200):
    arr = random_arr(rng, rng.randint(1, 8), 10, 4)
    assert max_length_backtrack(arr) == Solution().maxLength(arr) == max_length_k(arr, 1)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DP danh sách mask):**

```python
class Solution:
    def maxLength(self, arr: List[str]) -> int:
        dp = [0]
        for s in arr:
            m = 0
            for c in s:
                b = 1 << (ord(c) - 97)
                if m & b:
                    m = -1
                    break
                m |= b
            if m < 0:
                continue
            dp += [x | m for x in dp if x & m == 0]
        return max(bin(x).count("1") for x in dp)
```

**Checklist trước khi submit:**

- [ ] Loại chuỗi **tự có ký tự lặp** trước khi dùng.
- [ ] Kiểm tra `x & m == 0` trước khi ghép.
- [ ] Không sửa list khi đang duyệt nó; tạo list mới rồi nối.
- [ ] Đáp án tối thiểu là `0` (chuỗi rỗng) — `dp` khởi tạo `[0]`.
- [ ] Không dùng tham lam theo độ dài.
- [ ] Nếu backtracking với `set`, nhớ hoàn tác (hoặc dùng bitmask truyền theo giá trị).